In [1]:
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax


/Users/mthariqsultand/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
df = pd.read_csv('Data normalisasi tanpa labeling - Sheet1.csv')

# Select text column
df = df[['clean_textdisplay']].copy()

# Remove empty rows
df = df.dropna()

print(df.head())

                                   clean_textdisplay
0  tahun buat offsett baterai dan habis tahun gan...
1  ada ga ada mobil listrik pabrikan nikel terus ...
2  akhirnya persaingan negara china dengan electr...
3  akhirnya ttp beli electric vehicle sebenernya ...
4  aki mobil mobil aja mahalgmna ganti aki mobil ...


In [3]:
MODEL_NAME = "w11wo/indonesian-roberta-base-sentiment-classifier"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [4]:
label_mapping = {
    0: "negative",
    1: "neutral",
    2: "positive"
}


In [5]:
def predict_sentiment(text):

    # Tokenize text
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # Predict
    with torch.no_grad():
        outputs = model(**inputs)

    # Convert logits to probabilities
    scores = outputs.logits[0].numpy()
    probs = softmax(scores)

    # Get prediction
    predicted_class = probs.argmax()

    sentiment = label_mapping[predicted_class]
    confidence = probs[predicted_class]

    return sentiment, confidence


In [6]:
predictions = []
confidences = []

for text in df['clean_textdisplay']:

    sentiment, confidence = predict_sentiment(str(text))

    predictions.append(sentiment)
    confidences.append(confidence)


In [7]:
df['predicted_sentiment'] = predictions
df['confidence_score'] = confidences

# Save CSV
df.to_csv('hasil_sentiment_indobertweet.csv', index=False)
